# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [65]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [66]:
import os
import re
import requests
from pathlib import Path

pdf_url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
pdf_path = Path("Managing_Oneself_Drucker_HBR.pdf")

if not pdf_path.exists():
    r = requests.get(pdf_url, timeout=60)
    r.raise_for_status()
    pdf_path.write_bytes(r.content)

from pypdf import PdfReader

reader = PdfReader(str(pdf_path))
document_text = "\n".join((page.extract_text() or "") for page in reader.pages)
document_text = re.sub(r"\n{3,}", "\n\n", document_text).strip()

len(reader.pages), len(document_text)





(13, 51477)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [74]:
import os, json
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: Literal["Formal Academic Writing"]
    InputTokens: int
    OutputTokens: int

api_gateway_key = os.getenv("API_GATEWAY_KEY")

client_kwargs = {}
if api_gateway_key:
    client_kwargs = dict(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={"x-api-key": api_gateway_key},
    )

client = OpenAI(**client_kwargs) if client_kwargs else OpenAI()

developer_prompt = (
    "Return ONLY valid JSON for this schema with these exact keys:\n"
    "Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.\n"
    "Rules:\n"
    "- Do not include markdown or code fences.\n"
    "- Tone must be exactly: Formal Academic Writing.\n"
    "- Summary must be concise (<= 1000 tokens).\n"
    "- InputTokens and OutputTokens must be integers (set them to 0).\n"
)

user_prompt = f"""Document:
{document_text}

Create the JSON now.
"""

resp = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
    ],
)

raw_json = resp.output_text
generated_summary = ArticleSummary.model_validate(json.loads(raw_json))
generated_summary.InputTokens = int(getattr(resp.usage, "input_tokens", 0) or 0)
generated_summary.OutputTokens = int(getattr(resp.usage, "output_tokens", 0) or 0)

generated_summary


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [75]:
import os
from openai import OpenAI

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models.base_model import DeepEvalBaseLLM

api_gateway_key = os.getenv("API_GATEWAY_KEY")

client_kwargs = {}
if api_gateway_key:
    client_kwargs = dict(
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        api_key="any value",
        default_headers={"x-api-key": api_gateway_key},
    )

_eval_client = OpenAI(**client_kwargs) if client_kwargs else OpenAI()

class GatewayLLM(DeepEvalBaseLLM):
    def __init__(self, client: OpenAI, model_name: str = "gpt-4o-mini"):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        return self.client

    def get_model_name(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        r = self.client.responses.create(model=self.model_name, input=prompt)
        return r.output_text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def generate_raw_response(self, prompt: str, **kwargs):
        return self.generate(prompt), 0

    async def a_generate_raw_response(self, prompt: str, **kwargs):
        return await self.a_generate(prompt), 0

gateway_llm = GatewayLLM(_eval_client, model_name="gpt-4o-mini")

summary_text_v1 = generated_summary.Summary

test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_text_v1,
)

summarization_metric = SummarizationMetric(
    model=gateway_llm,
    assessment_questions=[
        "Does the summary accurately reflect the document without adding unsupported claims?",
        "Does it capture the central thesis and the most important supporting points?",
        "Is the summary concise and free of repetition or filler?",
        "Is the language clear and easy to follow for a professional audience?",
        "Does it avoid omitting key ideas that change the meaning of the article?",
    ],
)

coherence_metric = GEval(
    name="Coherence",
    model=gateway_llm,
    criteria="Evaluate clarity, logical flow, and organization of the summary.",
    evaluation_steps=[
        "Check that ideas are presented in a logical order.",
        "Check that sentences connect smoothly and avoid abrupt topic shifts.",
        "Check for ambiguous references (e.g., unclear pronouns).",
        "Check that each paragraph has a clear purpose.",
        "Give an overall coherence score based on clarity and structure.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

tonality_metric = GEval(
    name="Tonality",
    model=gateway_llm,
    criteria="Evaluate whether the summary matches the required tone: Formal Academic Writing.",
    evaluation_steps=[
        "Check for a formal, academic register (no slang or casual phrasing).",
        "Check for objective phrasing and precise word choice.",
        "Check for appropriate sentence structure and professionalism.",
        "Check that tone is consistent throughout the summary.",
        "Give an overall tonality score based on alignment with Formal Academic Writing.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

safety_metric = GEval(
    name="Safety",
    model=gateway_llm,
    criteria="Evaluate whether the summary is safe and avoids disallowed or harmful content.",
    evaluation_steps=[
        "Check for harassment, hate, or discriminatory language.",
        "Check for instructions facilitating wrongdoing or harm.",
        "Check for unsafe medical/legal/financial directives presented as certain.",
        "Check for privacy violations or sensitive personal data.",
        "Give an overall safety score based on the absence of unsafe content.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric],
)

evaluation_results_v1 = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_results_v1


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [77]:
developer_prompt_improve = (
    "You are an assistant that outputs ONLY the improved summary text. "
    "Use Formal Academic Writing. "
    "Do not add new facts not supported by the document. "
    "Keep it concise and logically structured."
)

user_prompt_improve = f"""
Document:
{document_text}

Original summary:
{generated_summary.Summary}

Evaluation feedback:
Summarization: {evaluation_results["SummarizationReason"]}
Coherence: {evaluation_results["CoherenceReason"]}
Tonality: {evaluation_results["TonalityReason"]}
Safety: {evaluation_results["SafetyReason"]}

Rewrite the summary to address the feedback. Output only the revised summary text.
"""

try:
    resp_v2 = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "developer", "content": developer_prompt_improve},
            {"role": "user", "content": user_prompt_improve},
        ],
    )
    improved_summary = resp_v2.output_text
except Exception:
    improved_summary = generated_summary.Summary

test_case_v2 = LLMTestCase(
    input=document_text,
    actual_output=improved_summary
)

evaluate(
    test_cases=[test_case_v2],
    metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric]
)

evaluation_results_v2 = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summararization_metric.reason if "summararization_metric" in globals() else summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

evaluation_results_v2


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

- The enhancement prompt explicitly reused the evaluation feedback to target weak areas (faithfulness/coverage, structure and tone consistency) while constraining the rewrite to the source document.
- The revised summary is expected to improve coherence (better organization and flow) and summarization quality (clearer focus on the document’s main claims) without changing the required tone.
- These controls are helpful but not sufficient alone: results still depend on the evaluator model and subtle factual drift can pass. Stronger guardrails would include multiple evaluators, tighter factuality checks (e.g., claim-to-source verification) and repeated runs to assess score stability.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
